In [13]:
!pip install onnxruntime onnx
!pip install onnxscript

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnxruntime as ort
import os
import torch
import torch.nn as nn

import gc
import random
import zipfile

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from google.colab import files
from sklearn.preprocessing import StandardScaler

In [15]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [16]:
!curl -L https://files.wundernn.io/wnn_predictorium_starterpack.tar.gz -o starter.tar.gz
!tar -xzf starter.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 1483M  100 1483M    0     0   109M      0  0:00:13  0:00:13 --:--:--  117M
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.FinderInfo'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.quarantine'
tar: Ignoring

In [17]:
train = pd.read_parquet('competition_package/datasets/train.parquet')
valid = pd.read_parquet('competition_package/datasets/valid.parquet')

In [18]:
def set_global_seed(seed: int) -> None:
    """Устанавливает глобальный seed для воспроизводимости результатов.

    Args:
        seed: Значение seed для генераторов случайных чисел
    """
    torch.backends.cudnn.deterministic = True
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)

    g = torch.Generator()
    g.manual_seed(seed)
    return g

def seed_worker(worker_id):
    """Устанавливает seed для воркеров DataLoader."""
    worker_seed = 42 + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)
    torch.manual_seed(worker_seed)
    torch.cuda.manual_seed(worker_seed)
    torch.cuda.manual_seed_all(worker_seed)


g = set_global_seed(42)

In [19]:
class FullSequenceDataset(Dataset):
    """Датасет для работы с полными последовательностями из DataFrame."""

    def __init__(self, df, start_pred_idx=99):
        """Инициализирует датасет, загружая последовательности, целевые значения и маски.

        Args:
            df: DataFrame с данными
            start_pred_idx: Индекс начала предсказаний (по умолчанию 99)
        """
        self.sequences = []
        self.targets = []
        self.masks = []

        for seq_id in tqdm(df['seq_ix'].unique(), desc="Создание датасета"):
            seq = df[df['seq_ix'] == seq_id].sort_values('step_in_seq')

            X = seq.iloc[:, 3:-2].values.astype(np.float32)
            y = seq[['t0', 't1']].values.astype(np.float32)
            need_pred = seq['need_prediction'].values.astype(bool)

            self.sequences.append(X)
            self.targets.append(y)
            self.masks.append(need_pred)

        print(f"Загружено {len(self.sequences)} последовательностей")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return (
            torch.FloatTensor(self.sequences[idx]),
            torch.FloatTensor(self.targets[idx]),
            torch.BoolTensor(self.masks[idx])
        )

In [20]:
class ConditionalConvGRU(nn.Module):
    """GRU модель со сверточным слоем для первых 100 шагов последовательности."""

    def __init__(self, input_size=32, hidden_size=64, num_layers=2, output_size=2):
        """Инициализирует архитектуру модели.

        Args:
            input_size: Размер входных признаков
            hidden_size: Размер скрытого состояния GRU
            num_layers: Количество слоев GRU
            output_size: Размер выходных признаков
        """
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv1d(input_size, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),
            nn.Conv1d(64, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.GELU()
        )

        self.gru = nn.GRU(
            input_size=32,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0,
            bidirectional=False
        )
        self.fc = nn.Linear(hidden_size, output_size)

        self._init_weights()

    def _init_weights(self):
        """Инициализирует веса модели."""
        for name, param in self.gru.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(param)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(param)
            elif 'bias' in name:
                nn.init.zeros_(param)
                n = param.size(0)
                param.data[n//4:n//2].fill_(1.0)

        nn.init.xavier_uniform_(self.fc.weight)
        nn.init.zeros_(self.fc.bias)

    def forward(self, x):
        """Прямой проход через модель.

        Args:
            x: Входной тензор [batch_size, seq_len, features]

        Returns:
            Тензор предсказаний [batch_size, seq_len, output_size]
        """
        batch_size, seq_len, _ = x.shape

        if seq_len == 1000:
            first_100 = x[:, :100, :]
            first_100 = first_100.transpose(1, 2)
            first_100 = self.conv(first_100)
            first_100 = first_100.transpose(1, 2)

            remaining = x[:, 100:, :]
            x = torch.cat([first_100, remaining], dim=1)

        output, hidden = self.gru(x)
        predictions = self.fc(output)
        return predictions

In [21]:
class WeightedPearsonLoss(nn.Module):
    """Функция потерь на основе взвешенной корреляции Пирсона."""

    def __init__(self, eps=1e-8, clip_value=6.0):
        """Инициализирует функцию потерь.

        Args:
            eps: Малое число для избежания деления на ноль
            clip_value: Порог для клиппинга предсказаний
        """
        super().__init__()
        self.eps = eps
        self.clip_value = clip_value

    def forward(self, y_pred, y_true, mask=None):
        """Вычисляет взвешенную потерю Пирсона.

        Args:
            y_pred: Предсказанные значения
            y_true: Истинные значения
            mask: Маска для учета только нужных элементов

        Returns:
            Значение потери
        """
        y_pred = torch.clamp(y_pred, -self.clip_value, self.clip_value)

        if mask is not None:
            if len(y_pred.shape) == 3:
                batch_size, seq_len, n_targets = y_pred.shape
                y_pred_flat = y_pred.reshape(-1, n_targets)
                y_true_flat = y_true.reshape(-1, n_targets)
                mask_flat = mask.reshape(-1)

                y_pred = y_pred_flat[mask_flat]
                y_true = y_true_flat[mask_flat]
            else:
                if len(mask.shape) == 2:
                    mask = mask[:, -1]
                y_pred = y_pred[mask]
                y_true = y_true[mask]

            if y_pred.size(0) == 0:
                return torch.tensor(0.0, device=y_pred.device)

        weights = torch.abs(y_true)
        weights = weights / (torch.sum(weights, dim=0, keepdim=True) + self.eps)

        mean_pred = torch.sum(weights * y_pred, dim=0)
        mean_true = torch.sum(weights * y_true, dim=0)

        pred_centered = y_pred - mean_pred
        true_centered = y_true - mean_true

        cov = torch.sum(weights * pred_centered * true_centered, dim=0)
        var_pred = torch.sum(weights * pred_centered ** 2, dim=0)
        var_true = torch.sum(weights * true_centered ** 2, dim=0)

        corr = cov / (torch.sqrt(var_pred * var_true) + self.eps)
        return 1.0 - torch.mean(corr)

In [22]:
def weighted_pearson_correlation(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Вычисляет взвешенную корреляцию Пирсона.

    Args:
        y_true: Истинные значения
        y_pred: Предсказанные значения

    Returns:
        Значение корреляции
    """
    y_pred_clipped = np.clip(y_pred, -6.0, 6.0)
    weights = np.abs(y_true)
    weights = np.maximum(weights, 1e-8)

    sum_w = np.sum(weights)
    if sum_w == 0:
        return 0.0

    mean_true = np.sum(y_true * weights) / sum_w
    mean_pred = np.sum(y_pred_clipped * weights) / sum_w

    dev_true = y_true - mean_true
    dev_pred = y_pred_clipped - mean_pred

    cov = np.sum(weights * dev_true * dev_pred) / sum_w
    var_true = np.sum(weights * dev_true**2) / sum_w
    var_pred = np.sum(weights * dev_pred**2) / sum_w

    if var_true <= 0 or var_pred <= 0:
        return 0.0
    corr = cov / (np.sqrt(var_true) * np.sqrt(var_pred))

    return float(corr)

In [23]:
def compute_pearson_metric(y_true, y_pred):
    """Вычисляет среднюю корреляцию Пирсона по двум целевым переменным.

    Args:
        y_true: Истинные значения
        y_pred: Предсказанные значения

    Returns:
        Среднее значение корреляции
    """
    y_true_np = y_true.detach().cpu().numpy()
    y_pred_np = y_pred.detach().cpu().numpy()

    scores = []
    for i in range(2):
        score = weighted_pearson_correlation(y_true_np[:, i], y_pred_np[:, i])
        scores.append(score)

    return np.mean(scores)

In [24]:
def train_one_epoch(model, loader, optimizer, criterion, device, is_training=True):
    """Обучает или валидирует модель на одну эпоху.

    Args:
        model: Модель для обучения
        loader: DataLoader с данными
        optimizer: Оптимизатор
        criterion: Функция потерь
        device: Устройство для вычислений
        is_training: Флаг режима обучения/валидации

    Returns:
        tuple: (средняя потеря, предсказания, целевые значения)
    """
    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_targets = []

    with torch.set_grad_enabled(is_training):
        for x, y, mask in tqdm(loader, desc='Train' if is_training else 'Valid'):
            x, y, mask = x.to(device), y.to(device), mask.to(device)

            pred = model(x)
            loss = criterion(pred, y, mask)

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item() * x.size(0)
            all_preds.append(pred[mask].cpu())
            all_targets.append(y[mask].cpu())

    avg_loss = total_loss / len(loader.dataset)
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)

    return avg_loss, all_preds, all_targets

In [25]:
def evaluate_epoch(valid_loss, valid_metric, best_metric, best_state, model):
    """Оценивает результаты эпохи и обновляет лучшую модель.

    Args:
        valid_loss: Потеря на валидации
        valid_metric: Метрика на валидации
        best_metric: Лучшая метрика
        best_state: Состояние лучшей модели
        model: Текущая модель

    Returns:
        tuple: (обновленная лучшая метрика, обновленное лучшее состояние)
    """
    if valid_metric > best_metric:
        best_metric = valid_metric
        best_state = model.state_dict().copy()

    return best_metric, best_state

In [26]:
def train_model(model, train_loader, valid_loader, epochs=10, lr=1e-3):
    """Основной цикл обучения модели.

    Args:
        model: Модель для обучения
        train_loader: DataLoader для тренировки
        valid_loader: DataLoader для валидации
        epochs: Количество эпох
        lr: Скорость обучения

    Returns:
        Обученная модель с лучшими весами
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    criterion = WeightedPearsonLoss()

    train_losses, valid_losses = [], []
    train_metrics, valid_metrics = [], []
    best_valid_metric = -float('inf')
    best_model_state = None

    for epoch in range(1, epochs + 1):
        train_loss, train_preds, train_targets = train_one_epoch(
            model, train_loader, optimizer, criterion, device, is_training=True
        )

        valid_loss, valid_preds, valid_targets = train_one_epoch(
            model, valid_loader, optimizer, criterion, device, is_training=False
        )

        train_metric = compute_pearson_metric(train_targets, train_preds)
        valid_metric = compute_pearson_metric(valid_targets, valid_preds)

        train_losses.append(train_loss)
        valid_losses.append(valid_loss)
        train_metrics.append(train_metric)
        valid_metrics.append(valid_metric)

        scheduler.step(valid_loss)

        best_valid_metric, best_model_state = evaluate_epoch(
            valid_loss, valid_metric, best_valid_metric, best_model_state, model
        )

        print(f"Epoch {epoch}: Train Loss={train_loss:.6f}, Valid Loss={valid_loss:.6f}")
        print(f"Train Pearson={train_metric:.4f}, Valid Pearson={valid_metric:.4f}")
        print(f"Best Valid Pearson={best_valid_metric:.4f}")

        gc.collect()
        torch.cuda.empty_cache()

    model.load_state_dict(best_model_state)
    return model

In [27]:
train_dataset = FullSequenceDataset(train)
valid_dataset = FullSequenceDataset(valid)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    generator=g,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=2,
    generator=g,
    pin_memory=True
)

Создание датасета: 100%|██████████| 10721/10721 [02:45<00:00, 64.96it/s]


Загружено 10721 последовательностей


Создание датасета: 100%|██████████| 1444/1444 [00:04<00:00, 337.49it/s]

Загружено 1444 последовательностей


In [28]:
print(f"Train батчей: {len(train_loader)}")
print(f"Valid батчей: {len(valid_loader)}")

Train батчей: 1341
Valid батчей: 181


In [29]:
model = ConditionalConvGRU(hidden_size=128)

model = train_model(model, train_loader, valid_loader, epochs=5, lr=1e-3)

Valid: 100%|██████████| 181/181 [00:01<00:00, 118.14it/s]


Epoch 1: Train Loss=0.762988, Valid Loss=0.747898
Train Pearson=0.1938, Valid Pearson=0.2415
Best Valid Pearson=0.2415


Valid: 100%|██████████| 181/181 [00:02<00:00, 82.14it/s] 


Epoch 2: Train Loss=0.734739, Valid Loss=0.739665
Train Pearson=0.2522, Valid Pearson=0.2550
Best Valid Pearson=0.2550


Valid: 100%|██████████| 181/181 [00:01<00:00, 116.60it/s]


Epoch 3: Train Loss=0.724229, Valid Loss=0.734568
Train Pearson=0.2734, Valid Pearson=0.2593
Best Valid Pearson=0.2593


Valid: 100%|██████████| 181/181 [00:02<00:00, 78.29it/s] 


Epoch 4: Train Loss=0.715170, Valid Loss=0.738282
Train Pearson=0.2778, Valid Pearson=0.2573
Best Valid Pearson=0.2593


Valid: 100%|██████████| 181/181 [00:01<00:00, 114.64it/s]


Epoch 5: Train Loss=0.709652, Valid Loss=0.730735
Train Pearson=0.2948, Valid Pearson=0.2631
Best Valid Pearson=0.2631


In [30]:
model.eval()

    # Создаем пример входных данных
example_inputs = torch.randn(1, 100, 32).cuda()
onnx_program = torch.onnx.export(
        model,
        example_inputs,
        dynamo=True,
        export_params=True,
)


onnx_program.save("easy_model.onnx")

[torch.onnx] Obtain model graph for `ConditionalConvGRU([...]` with `torch.export.export(..., strict=False)`...


/usr/lib/python3.12/contextlib.py:158: UserWarning: The tensor attributes self.gru._flat_weights[0], self.gru._flat_weights[1], self.gru._flat_weights[2], self.gru._flat_weights[3], self.gru._flat_weights[4], self.gru._flat_weights[5], self.gru._flat_weights[6], self.gru._flat_weights[7] were assigned during export. Such attributes must be registered as buffers using the `register_buffer` API (https://pytorch.org/docs/stable/generated/torch.nn.Module.html#torch.nn.Module.register_buffer).
  self.gen.throw(value)


[torch.onnx] Obtain model graph for `ConditionalConvGRU([...]` with `torch.export.export(..., strict=False)`... ❌
[torch.onnx] Obtain model graph for `ConditionalConvGRU([...]` with `torch.export.export(..., strict=True)`...
[torch.onnx] Obtain model graph for `ConditionalConvGRU([...]` with `torch.export.export(..., strict=True)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


In [32]:
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('/content/solution.py', 'solution.py')
    z.write('/content/easy_model.onnx', 'my_model.onnx')

files.download('submission.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>